# MDG — adversarial attacks on Colab

Fallback for when the VMs aren't available. Attacks the **complete test
split** of every dataset — no example cap, no query budget.

Colab kills sessions, so this is built to resume: results live on Drive and
every finished unit is skipped next time. A timeout costs at most one unit.

**Run cells 1–5 once per session, then cell 6 repeatedly.**

Use a GPU runtime: *Runtime → Change runtime type → T4*. Attacks are
thousands of model queries per example, which is exactly what a GPU is for.

## 1. What hardware did we get

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo 'NO GPU — switch the runtime type'
!nproc && free -g | head -2

## 2. Mount Drive

Everything that matters is written here, so a dead session loses nothing.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

MDG_DRIVE = '/content/drive/MyDrive/mdg'
import os; os.makedirs(MDG_DRIVE, exist_ok=True)
os.environ['MDG_DRIVE'] = MDG_DRIVE
print('results will persist in', MDG_DRIVE)

## 3. HuggingFace token

`getpass` so the token is never written into the notebook.

In [ ]:
import getpass, pathlib
p = pathlib.Path.home() / '.hf_token'
if not p.exists():
    p.write_text(getpass.getpass('HF token: ')); p.chmod(0o600)
print('token ready')

## 4. Clone the repo

In [ ]:
import os
if not os.path.isdir('/content/mdg/.git'):
    !git clone -q https://github.com/namesarnav/mdg.git /content/mdg
else:
    !cd /content/mdg && git pull -q
os.chdir('/content/mdg')
!git log --oneline -1

## 5. Setup — deps, data, 33 models, attack cache

Takes 10–20 minutes the first time. The model and asset caches go to Drive,
so later sessions skip the download entirely.

In [ ]:
!MDG_DRIVE=$MDG_DRIVE bash colab/setup.sh

## 6. Run

Works for N hours then stops cleanly between units. **Re-run this cell to
continue** — in the same session or a new one.

Cheapest work goes first, so a short session still finishes whole units
instead of stalling halfway through PSO on the largest dataset.

Running several Colab accounts? Give each a different `SHARD`:
`SHARD=0/3`, `SHARD=1/3`, `SHARD=2/3`.

In [ ]:
!MDG_DRIVE=$MDG_DRIVE SHARD=0/1 bash colab/run.sh 11

## 7. Progress

In [ ]:
!cd /content/mdg && python3 mdg/hpc/status.py 2>/dev/null | head -40
!echo '--- recent ---' && tail -20 $MDG_DRIVE/logs/colab-run.log 2>/dev/null

## 8. When everything is done

Merges the row shards and writes the final table.

In [ ]:
!cd /content/mdg && python3 mdg/scripts/merge_shards.py
!cd /content/mdg && python3 -m mdg.scripts.consolidate_results
!cp /content/mdg/mdg/results/consolidated.csv $MDG_DRIVE/ && echo 'saved to Drive'

---

### If the session dies mid-run

Nothing is lost. Re-run cells 2–5 (fast, everything is cached on Drive)
then cell 6. Finished units are skipped.

### Keep `ROW_SHARDS` at 64

The shard count is baked into output filenames (`<recipe>.sh3of64.jsonl`).
A cell attacked at a different count here cannot be merged with the same
cell attacked on a VM.